# Lesson 5.1 — IAM Deep Dive: Roles, SCPs, Identity Center & Organizations

**Level: Intermediate** | **Exam:** SOA-C03 Domain 4, Task 4.1; SAP-C02 Domain 1, Tasks 1.2 & 1.4

## Learning Objectives
- Implement IAM roles for cross-account access and service federation
- Design Service Control Policies (SCPs) in AWS Organizations
- Configure IAM Identity Center (SSO) for federated access
- Audit IAM permissions using IAM Access Analyzer and policy simulator
- Apply permission boundaries to limit developer permissions

_Reference: [IAM Policy Reference](https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_elements.html) | [AWS Organizations SCPs](https://docs.aws.amazon.com/organizations/latest/userguide/orgs_manage_policies_scps.html)_

## IAM Policy Evaluation Logic

When an API call is made, AWS evaluates policies in this order:

```
1. DENY? → Check for explicit Deny in any policy → DENY if found
2. SCP? → Check Organizations SCP allows the action → DENY if not allowed
3. Resource policy? → Check if resource policy allows → ALLOW if yes
4. Identity policy? → Check IAM role/user/group policies → ALLOW if yes
5. Permission boundary? → Is this action within the boundary? → DENY if not
6. Session policy? → For assumed roles, check session policy → DENY if not
Default: DENY (implicit deny)
```

**Key rule: Explicit Deny always wins over Allow.**

## Service Control Policies (SCPs)

SCPs are **guardrails** applied at the AWS Organizations level:
- Limit what member accounts can do — even if their IAM policies say Allow
- SCPs do NOT grant permissions — they only restrict
- Applied to: OU (Organizational Unit), Account, or Root
- Management account is **exempt** from SCPs

**Common SCP examples:**
```json
// Deny leaving Organizations
{"Effect": "Deny", "Action": "organizations:LeaveOrganization", "Resource": "*"}
// Restrict to approved regions
{"Effect": "Deny", "Action": "*", "Resource": "*",
 "Condition": {"StringNotEquals": {"aws:RequestedRegion": ["us-east-1", "eu-west-1"]}}}
// Require MFA for sensitive actions
{"Effect": "Deny", "Action": ["iam:*"], "Resource": "*",
 "Condition": {"BoolIfExists": {"aws:MultiFactorAuthPresent": "false"}}}
```

## Permission Boundaries

Permission Boundaries limit the **maximum permissions** an IAM entity can have:
```
Effective Permissions = IAM Policy ∩ Permission Boundary
```
Use case: Give developers permission to create IAM roles, but only roles that can't do more than the developer can do.

## IAM Identity Center (formerly SSO)
- Central access management across all accounts in AWS Organizations
- Connect to identity providers: Okta, Azure AD, Active Directory
- Users get temporary credentials (no long-term access keys)
- Permission Sets = IAM policies applied when a user accesses an account

In [ ]:
import boto3, json

iam = boto3.client('iam')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Cross-Account Role — allow Account B to assume a role in Account A

# In Account A (this account), create a role that Account B can assume
DEVELOPER_ACCOUNT = '123456789012'  # Replace with actual account B ID

cross_account_trust = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {
                'AWS': f'arn:aws:iam::{DEVELOPER_ACCOUNT}:root'  # Any IAM entity in Account B
            },
            'Action': 'sts:AssumeRole',
            'Condition': {
                'Bool': {'aws:MultiFactorAuthPresent': 'true'},    # Require MFA
                'StringEquals': {'sts:ExternalId': 'unique-secret-id-12345'}  # Prevent confused deputy
            }
        }
    ]
}

print('Cross-Account Role Trust Policy:')
print(json.dumps(cross_account_trust, indent=2))
print()
print('In Account B, an IAM user/role can now assume this role with:')
print('aws sts assume-role \\')
print(f'  --role-arn arn:aws:iam::{account_id}:role/CrossAccountReadRole \\')
print('  --role-session-name MySession \\')
print('  --serial-number arn:aws:iam::ACCOUNTB:mfa/user \\')
print('  --token-code 123456')

In [ ]:
# Common SCP (Service Control Policy) patterns
# These would be applied in AWS Organizations, not directly here

scps = {
    'DenyLeavingOrganizations': {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'DenyLeaveOrg',
            'Effect': 'Deny',
            'Action': 'organizations:LeaveOrganization',
            'Resource': '*'
        }]
    },
    
    'RestrictToApprovedRegions': {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'DenyUnapprovedRegions',
            'Effect': 'Deny',
            'Action': '*',
            'Resource': '*',
            'Condition': {
                'StringNotEquals': {
                    'aws:RequestedRegion': ['us-east-1', 'us-west-2', 'eu-west-1']
                },
                'ArnNotLike': {
                    'aws:PrincipalARN': [f'arn:aws:iam::*:role/BreakGlassRole']  # Exempt break-glass
                }
            }
        }]
    },
    
    'RequireMFAForSensitiveActions': {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'DenyWithoutMFA',
            'Effect': 'Deny',
            'Action': ['iam:*', 'cloudtrail:Stop*', 'cloudtrail:Delete*'],
            'Resource': '*',
            'Condition': {
                'BoolIfExists': {'aws:MultiFactorAuthPresent': 'false'}
            }
        }]
    }
}

print('Common SCP Patterns:')
for name, policy in scps.items():
    print(f'\n{name}:')
    stmt = policy['Statement'][0]
    print(f'  Effect: {stmt["Effect"]}')
    print(f'  Action: {stmt["Action"]}')
    if 'Condition' in stmt:
        print(f'  Condition: {list(stmt["Condition"].keys())}')
    print(f'  Purpose: protects against {name}')

In [ ]:
# IAM Access Analyzer — find unintended external access to your resources

access_analyzer = boto3.client('accessanalyzer', region_name='us-east-1')

try:
    analyzers = access_analyzer.list_analyzers()['analyzers']
    if analyzers:
        print(f'IAM Access Analyzers ({len(analyzers)}):')
        for a in analyzers:
            print(f'  {a["name"]:30} Status: {a["status"]} Scope: {a["type"]}')
        
        # List findings
        findings = access_analyzer.list_findings(analyzerArn=analyzers[0]['arn'])['findings']
        print(f'\nFindings ({len(findings)}):')
        for f in findings[:5]:
            print(f'  {f["resourceType"]:30} → {f["condition"]} [{f["status"]}]')
    else:
        print('No Access Analyzers found. Create one to detect unintended external access.')
        print('aws accessanalyzer create-analyzer --analyzer-name account-analyzer --type ACCOUNT')
except Exception as e:
    print(f'Access Analyzer: {e}')

## Exercise

In [ ]:
# TODO: Write an IAM policy audit function that:
# 1. Lists all IAM users with console access (PasswordLastUsed)
# 2. Identifies users who haven't logged in for 90+ days
# 3. Identifies users with admin policies attached (AdministratorAccess)
# 4. Prints a risk report

import boto3
from datetime import datetime, timezone, timedelta

def iam_security_audit():
    iam = boto3.client('iam')
    issues = []
    
    # YOUR CODE HERE
    
    return issues

results = iam_security_audit()
print(f'IAM Security Audit: {len(results)} issues found')
for issue in results:
    print(f'  [{issue.get("severity", "MEDIUM")}] {issue["user"]}: {issue["issue"]}')

## Summary
- IAM evaluation: SCP → Permission Boundary → Identity Policy → Resource Policy (Explicit Deny always wins)
- **SCPs** restrict what accounts in Organizations can do — even admins in member accounts
- **Permission Boundaries** cap the maximum effective permissions (useful for developer self-service)
- **IAM Identity Center**: Federated SSO for multi-account access, preferred over long-term IAM users
- **IAM Access Analyzer**: Identifies resources accessible from outside the account

---
**Next:** [Lesson 5.2 — Encryption at Rest & In Transit: KMS, ACM & Secrets Manager](lesson-5.2-encryption-secrets.ipynb)